# 03 — Evolutionary scores (ESM-2 650M)

dualde infers a single sequence model: **ESM-2 650M** (`esm2_t33_650M_UR50D`).
Inference runs **only if a CUDA GPU is available**; otherwise this notebook
loads the precomputed table.

**EVmutation** scores and the **coupling graph** are not computed here. Obtain
both from the [EVcouplings webserver](https://evcouplings.org). For the NUDT15
example they are already included in `scores/evolutionary.csv` and
`couplings/b0.5_CouplingScoresCompared_all.csv`.

## Resolve scores (GPU scan or cached table)

In [1]:
from IPython.display import display
import pandas as pd

from dualde import load_example
from dualde.mapping import read_fasta
from dualde.scores import ESM2_650M_COLUMN, cuda_is_available, run_esm2_650m

EXAMPLE, cfg = load_example()
OUT = EXAMPLE / "outputs"
OUT.mkdir(parents=True, exist_ok=True)
CACHE = EXAMPLE / "scores" / "evolutionary.csv"

cached = pd.read_csv(CACHE)

if cuda_is_available():
    print("CUDA GPU detected; running ESM-2 650M.")
    _, sequence = read_fasta(EXAMPLE / f"{cfg['protein']}.fasta")
    try:
        evo = run_esm2_650m(sequence, protein_id=cfg["protein"])
        if "EVmutation" in cached.columns:
            evo = evo.merge(
                cached[["mutation", "EVmutation"]].drop_duplicates(),
                on="mutation",
                how="left",
            )
    except ImportError as err:
        print(err)
        print("Falling back to the precomputed table.")
        evo = cached
else:
    print("No CUDA GPU; loading precomputed ESM-2 650M scores.")
    evo = cached

evo.to_csv(OUT / "evolutionary.csv", index=False)
print("wrote", OUT / "evolutionary.csv")

No CUDA GPU; loading precomputed ESM-2 650M scores.
wrote /lustre/scratch125/gengen/projects_v2/sgb_enzyme_design/evedesign_prelim/dualde/examples/NUDT15_HUMAN/outputs/evolutionary.csv


## Inspect the score table

In [2]:
print("shape:", evo.shape)
print("columns:", list(evo.columns))
display(evo.head())

shape: (3444, 10)
columns: ['pos', 'ref', 'aa_to', 'mutation', 'ESM2_35M', 'ESM2_650M', 'ESM2_3B', 'EVmutation2', 'EVmutation', 'EVE']


,pos,ref,aa_to,mutation,ESM2_35M,ESM2_650M,ESM2_3B,EVmutation2,EVmutation,EVE
0,1,M,-,M1-,NaN,NaN,NaN,10.905360,NaN,NaN
1,1,M,A,M1A,-6.611199,-5.126851,-5.808349,0.547099,NaN,NaN
2,1,M,C,M1C,-8.421957,-7.465296,-9.379663,-2.163146,NaN,NaN
3,1,M,D,M1D,-7.341832,-6.857292,-8.526442,1.033458,NaN,NaN
4,1,M,E,M1E,-7.009660,-6.189185,-8.517876,0.361666,NaN,NaN


In [3]:
esm_cols = [c for c in evo.columns if str(c).startswith("ESM2")]
print("ESM-2 columns (dualde uses ESM2_650M):", esm_cols)
print("missingness:")
cols = [c for c in [ESM2_650M_COLUMN, "EVmutation"] if c in evo.columns]
display(evo[cols].isna().mean().to_frame("frac_na"))
if ESM2_650M_COLUMN in evo.columns:
    display(evo[[ESM2_650M_COLUMN]].describe().T)
else:
    print("WARNING: ESM2_650M is absent from this table.")

ESM-2 columns (dualde uses ESM2_650M): ['ESM2_35M', 'ESM2_650M', 'ESM2_3B']
missingness:


,frac_na
ESM2_650M,0.047619
EVmutation,0.299361


,count,mean,std,min,25%,50%,75%,max
ESM2_650M,3280.0,-3.209501,3.592515,-15.31274,-5.428868,-2.240642,-0.377204,10.026585


In [4]:
site_pos = int(evo["pos"].median()) if "pos" in evo.columns else None
if site_pos is not None:
    site_cols = ["pos", "ref", "aa_to", "mutation"] + [
        c for c in [ESM2_650M_COLUMN, "EVmutation"] if c in evo.columns
    ]
    site = evo.loc[evo["pos"] == site_pos, site_cols]
    print(f"position {site_pos} (n = {len(site)})")
    display(site.head(10))

position 82 (n = 21)


,pos,ref,aa_to,mutation,ESM2_650M,EVmutation
1701,82,N,-,N82-,NaN,NaN
1702,82,N,A,N82A,-6.106047,-7.087401
1703,82,N,C,N82C,-6.344226,-8.769187
1704,82,N,D,N82D,-2.391729,-4.317093
1705,82,N,E,N82E,-2.580917,-4.769653
1706,82,N,F,N82F,-8.684932,-7.711321
1707,82,N,G,N82G,-5.211681,-7.321593
1708,82,N,H,N82H,-4.942822,-6.691925
1709,82,N,I,N82I,-8.569142,-8.769187
1710,82,N,K,N82K,-7.126819,-8.735095


## EVcouplings webserver (new proteins)

1. Submit the FASTA at [evcouplings.org](https://evcouplings.org).
2. Merge the single-mutant / EVmutation table onto `mutation` (for example `Q144A`) as column `EVmutation`.
3. Place `*CouplingScoresCompared_all.csv` (`i`, `j`, `probability`, `dist`) where notebook 06 can read it.

Optional GPU install: `pip install -e ".[evo]"` and `export HF_HOME=$PWD/.cache/huggingface`.